# A Generic Ingestion Framework

## 1. Setup

In [1]:
import json
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import DataFrame, SparkSession, functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, LongType, StringType, StructField,
    StructType, TimestampType,
)
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

In [ ]:
# Create spark session
builder = (
    SparkSession.builder
    .appName("Lab1-Generic-Ingestion")
    .master("local[*]")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")


:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0acefc07-c4b7-45d1-b867-ee0cf1a14c43;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 52506)
Traceback (most recent call last):
  File "/Users/rexchang/.local/share/uv/python/cpython-3.14.6-macos-aarch64-none/lib/python3.14/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rexchang/.local/share/uv/python/cpython-3.14.6-macos-aarch64-none/lib/python3.14/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rexchang/.local/share/uv/python/cpython-3.14.6-macos-aarch64-none/lib/python3.14/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rexchang/.local/share/uv/python/cpython-3.14.6-macos-aarch64-none/lib/python3.14/

In [3]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a parent directory containing data/raw")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FAILED_DIR = PROJECT_ROOT / "data" / "failed"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
for directory in (PROCESSED_DIR, FAILED_DIR, METADATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Spark version:", spark.version)
print("Project root:", PROJECT_ROOT)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)
print("Failed:", FAILED_DIR)
print("Metadata:", METADATA_DIR)

Spark version: 4.2.0
Project root: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing
Raw: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/raw
Processed: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed
Failed: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/failed
Metadata: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/metadata


## 2. Dataset configuration and expected input schemas

Generic behavior is configuration-driven. Each dataset defines its source format, expected raw schema, schema version, transformation, required processed fields, primary key, validation rules, output path, and partition columns.

In [4]:
TAXI_RAW_TYPES = {
    "VendorID": {"int"},
    "tpep_pickup_datetime": {"timestamp", "timestamp_ntz"},
    "tpep_dropoff_datetime": {"timestamp", "timestamp_ntz"},
    "passenger_count": {"bigint", "int"},
    "trip_distance": {"double"},
    "RatecodeID": {"bigint", "int"},
    "store_and_fwd_flag": {"string"},
    "PULocationID": {"int"},
    "DOLocationID": {"int"},
    "payment_type": {"bigint", "int"},
    "fare_amount": {"double"},
    "extra": {"double"},
    "mta_tax": {"double"},
    "tip_amount": {"double"},
    "tolls_amount": {"double"},
    "improvement_surcharge": {"double"},
    "total_amount": {"double"},
    "congestion_surcharge": {"double"},
    "Airport_fee": {"double"},
}

WEATHER_RAW_TYPES = {
    "year": {"int"}, "month": {"int"}, "day": {"int"}, "hour": {"int"},
    "temp": {"double"}, "temp_source": {"string"},
    "rhum": {"int", "bigint"}, "rhum_source": {"string"},
    "prcp": {"double"}, "prcp_source": {"string"},
    "snwd": {"string", "double"}, "snwd_source": {"string"},
    "wdir": {"int", "bigint"}, "wdir_source": {"string"},
    "wspd": {"double"}, "wspd_source": {"string"},
    "wpgt": {"string", "double"}, "wpgt_source": {"string"},
    "pres": {"double"}, "pres_source": {"string"},
    "cldc": {"int", "bigint"}, "cldc_source": {"string"},
    "coco": {"int", "bigint"}, "coco_source": {"string"},
}

AIR_RAW_COLUMNS = [
    "State Code", "County Code", "Site Num", "Parameter Code", "POC",
    "Latitude", "Longitude", "Datum", "Parameter Name", "Date Local",
    "Time Local", "Date GMT", "Time GMT", "Sample Measurement",
    "Units of Measure", "MDL", "Uncertainty", "Qualifier", "Method Type",
    "Method Code", "Method Name", "State Name", "County Name",
    "Date of Last Change",
]
AIR_RAW_TYPES = {column: {"string"} for column in AIR_RAW_COLUMNS}
ZONE_RAW_TYPES = {
    "LocationID": {"int"}, "Borough": {"string"},
    "Zone": {"string"}, "service_zone": {"string"},
}

In [5]:
DATASETS = {
    "taxi_trips": {
        "format": "parquet",
        "source_path": str(RAW_DIR / "yellow_tripdata_2024-*.parquet"),
        "expected_raw_types": TAXI_RAW_TYPES,
        "schema_version": "1.0.0",
        "transformer": "taxi",
        "primary_key": ["trip_id"],
        "required_columns": [
            "trip_id", "pickup_datetime", "dropoff_datetime",
            "pickup_location_id", "dropoff_location_id",
        ],
        "partition_by": ["pickup_year", "pickup_month"],
    },
    "weather": {
        "format": "csv",
        "source_path": str(RAW_DIR / "weather.csv"),
        "read_options": {"header": "true", "inferSchema": "true"},
        "expected_raw_types": WEATHER_RAW_TYPES,
        "schema_version": "1.0.0",
        "transformer": "weather",
        "primary_key": ["observation_timestamp"],
        "required_columns": ["observation_timestamp"],
        "partition_by": [],
    },
    "air_quality": {
        "format": "csv",
        "source_path": str(RAW_DIR / "hourly_88101_2024.csv"),
        "read_options": {"header": "true", "inferSchema": "false"},
        "expected_raw_types": AIR_RAW_TYPES,
        "schema_version": "1.0.0",
        "transformer": "air_quality",
        "primary_key": ["observation_id"],
        "required_columns": [
            "observation_id", "station_id", "parameter_code",
            "observation_timestamp", "observation_timestamp_utc",
            "sample_measurement",
        ],
        "partition_by": ["observation_year", "observation_month"],
    },
    "taxi_zones": {
        "format": "csv",
        "source_path": str(RAW_DIR / "taxi_zone_lookup.csv"),
        "read_options": {"header": "true", "inferSchema": "true"},
        "expected_raw_types": ZONE_RAW_TYPES,
        "schema_version": "1.0.0",
        "transformer": "taxi_zones",
        "primary_key": ["location_id"],
        "required_columns": ["location_id", "borough", "zone"],
        "partition_by": [],
    },
}

In [6]:
for dataset_name, config in DATASETS.items():
    config["processed_path"] = str(PROCESSED_DIR / dataset_name)
    config["failed_path"] = str(FAILED_DIR / dataset_name)

METADATA_PATH = str(METADATA_DIR / "ingestion_runs")
DATASETS

{'taxi_trips': {'format': 'parquet',
  'source_path': '/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/raw/yellow_tripdata_2024-*.parquet',
  'expected_raw_types': {'VendorID': {'int'},
   'tpep_pickup_datetime': {'timestamp', 'timestamp_ntz'},
   'tpep_dropoff_datetime': {'timestamp', 'timestamp_ntz'},
   'passenger_count': {'bigint', 'int'},
   'trip_distance': {'double'},
   'RatecodeID': {'bigint', 'int'},
   'store_and_fwd_flag': {'string'},
   'PULocationID': {'int'},
   'DOLocationID': {'int'},
   'payment_type': {'bigint', 'int'},
   'fare_amount': {'double'},
   'extra': {'double'},
   'mta_tax': {'double'},
   'tip_amount': {'double'},
   'tolls_amount': {'double'},
   'improvement_surcharge': {'double'},
   'total_amount': {'double'},
   'congestion_surcharge': {'double'},
   'Airport_fee': {'double'}},
  'schema_version': '1.0.0',
  'transformer': 'taxi',
  'primary_key': ['trip_id'],
  'required_columns': ['t

## 3. Generic source loading, schema validation, and naming

In [7]:
# Convert column names to a snake_case format
def to_snake_case(name: str) -> str:
    value = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    value = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", value)
    value = re.sub(r"[^A-Za-z0-9]+", "_", value)
    return re.sub(r"_+", "_", value).strip("_").lower()

# Standardize column names to snake_case, except for columns that start with an underscore
def standardize_column_names(df: DataFrame) -> DataFrame:
    renamed = df
    for old_name in df.columns:
        new_name = old_name if old_name.startswith("_") else to_snake_case(old_name)
        renamed = renamed.withColumnRenamed(old_name, new_name)
    return renamed

# Read a dataset from the source path specified in the config, using the format and read options specified in the config
def read_source(config: dict) -> DataFrame:
    reader = spark.read.format(config["format"])
    for option_name, option_value in config.get("read_options", {}).items():
        reader = reader.option(option_name, option_value)
    return reader.load(config["source_path"])

# Validate that the DataFrame's schema matches the expected raw types specified in the config
def validate_input_schema(df: DataFrame, expected_raw_types: dict) -> str:
    actual = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    expected_columns = set(expected_raw_types)
    actual_columns = set(actual)
    missing_columns = sorted(expected_columns - actual_columns)
    unexpected_columns = sorted(actual_columns - expected_columns)
    type_mismatches = {
        column: {"actual": actual[column], "allowed": sorted(expected_raw_types[column])}
        for column in sorted(expected_columns & actual_columns)
        if actual[column] not in expected_raw_types[column]
    }
    if missing_columns or unexpected_columns or type_mismatches:
        raise ValueError(json.dumps({
            "missing_columns": missing_columns,
            "unexpected_columns": unexpected_columns,
            "type_mismatches": type_mismatches,
        }, indent=2))
    return json.dumps(actual, sort_keys=True)

## 4. Dataset-specific transformations

These functions contain only the transformations that cannot be inferred from the generic configuration.

In [8]:
# Return a condition that checks whether it can be safely used as a primary key
def require_all(columns):
    condition = F.lit(True)
    for column in columns:
        condition = condition & F.col(column).isNotNull() & (F.trim(F.col(column).cast("string")) != "")
    return condition

# Compute a stable hash of the specified columns, treating null values as "<null>"
def stable_hash(columns):
    return F.sha2(
        F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in columns]),
        256,
    )

In [9]:
# Standardize the taxi DataFrame
def transform_taxi(df: DataFrame) -> DataFrame:
    df = standardize_column_names(df)
    
    # Unified data type
    df = ( 
        df
        .withColumnRenamed("pu_location_id", "pickup_location_id")
        .withColumnRenamed("do_location_id", "dropoff_location_id")
        .withColumnRenamed("ratecode_id", "rate_code_id")
        .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
        .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
        .withColumn("vendor_id", F.col("vendor_id").cast(LongType()))
        .withColumn("pickup_datetime", F.col("pickup_datetime").cast(TimestampType()))
        .withColumn("dropoff_datetime", F.col("dropoff_datetime").cast(TimestampType()))
        .withColumn("passenger_count", F.col("passenger_count").cast(IntegerType()))
        .withColumn("rate_code_id", F.col("rate_code_id").cast(LongType()))
        .withColumn("pickup_location_id", F.col("pickup_location_id").cast(IntegerType()))
        .withColumn("dropoff_location_id", F.col("dropoff_location_id").cast(IntegerType()))
        .withColumn("payment_type", F.col("payment_type").cast(LongType()))
    )
    # Create trip_id, trip_duration_minutes, pickup_date, pickup_hour, pickup_year, and pickup_month
    for column in [
        "trip_distance", "fare_amount", "extra", "mta_tax", "tip_amount",
        "tolls_amount", "improvement_surcharge", "total_amount",
        "congestion_surcharge", "airport_fee",
    ]:
        df = df.withColumn(column, F.col(column).cast(DoubleType()))
    key_columns = [
        "vendor_id", "pickup_datetime", "dropoff_datetime",
        "pickup_location_id", "dropoff_location_id", "trip_distance", "total_amount",
    ]
    return (
        df
        .withColumn("trip_id", F.when(require_all(key_columns), stable_hash(key_columns)))
        .withColumn("trip_duration_minutes", F.expr("timestampdiff(SECOND, pickup_datetime, dropoff_datetime)") / F.lit(60.0))
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
        .withColumn("pickup_hour", F.date_trunc("hour", "pickup_datetime"))
        .withColumn("pickup_year", F.year("pickup_datetime").cast(IntegerType()))
        .withColumn("pickup_month", F.month("pickup_datetime").cast(IntegerType()))
    )

In [10]:
# Standardize the weather DataFrame
def transform_weather(df: DataFrame) -> DataFrame:
    df = standardize_column_names(df)
    # Unified data type
    for column in ["year", "month", "day", "hour", "rhum", "wdir", "cldc", "coco"]:
        df = df.withColumn(column, F.col(column).cast(IntegerType()))
    for column in ["temp", "prcp", "snwd", "wspd", "wpgt", "pres"]:
        df = df.withColumn(column, F.col(column).cast(DoubleType()))
    # Create observation_timestamp and observation_date
    return (
        df
        .withColumn("observation_timestamp", F.make_timestamp("year", "month", "day", "hour", F.lit(0), F.lit(0)).cast(TimestampType()))
        .withColumn("observation_date", F.to_date("observation_timestamp"))
    )

In [11]:
# Standardize the air quality DataFrame
def transform_air_quality(df: DataFrame) -> DataFrame:
    df = standardize_column_names(df)
    # Unified data type
    for column in ["latitude", "longitude", "sample_measurement", "mdl", "uncertainty"]:
        df = df.withColumn(column, F.col(column).cast(DoubleType()))
    # Create two timestamp
    df = (
        df
        .withColumn("observation_timestamp", F.to_timestamp(F.concat_ws(" ", "date_local", "time_local"), "yyyy-MM-dd HH:mm"))
        .withColumn("observation_timestamp_utc", F.to_timestamp(F.concat_ws(" ", "date_gmt", "time_gmt"), "yyyy-MM-dd HH:mm"))
        .withColumn("date_of_last_change", F.to_date("date_of_last_change", "yyyy-MM-dd"))
        .withColumn("station_id", F.concat_ws("-", "state_code", "county_code", "site_num", "parameter_code", "poc"))
    )
    # Create station_id, observation_id
    key_columns = [
        "state_code", "county_code", "site_num", "parameter_code",
        "poc", "observation_timestamp", "method_code",
    ]
    # Create partition field
    return (
        df
        .withColumn("observation_id", F.when(require_all(key_columns), stable_hash(key_columns)))
        .withColumn("observation_date", F.to_date("observation_timestamp"))
        .withColumn("observation_year", F.year("observation_timestamp").cast(IntegerType()))
        .withColumn("observation_month", F.month("observation_timestamp").cast(IntegerType()))
        .drop("date_local", "time_local", "date_gmt", "time_gmt")
    )

In [12]:
# Standardize the taxi zones DataFrame
def transform_taxi_zones(df: DataFrame) -> DataFrame:
    df = standardize_column_names(df).withColumnRenamed("location_id", "location_id")
    return (
        df
        .withColumn("location_id", F.col("location_id").cast(IntegerType()))
        .withColumn("borough", F.trim("borough"))
        .withColumn("zone", F.trim("zone"))
        .withColumn("service_zone", F.trim("service_zone"))
    )

In [13]:
TRANSFORMERS = {
    "taxi": transform_taxi,
    "weather": transform_weather,
    "air_quality": transform_air_quality,
    "taxi_zones": transform_taxi_zones,
}

## 5. Generic and dataset-specific validation

The generic layer checks missing required fields and duplicate primary keys. Dataset-specific rules check timestamp consistency and numeric domains. Duplicate handling retains one record in `processed` and writes later occurrences to `failed`.

In [14]:
def missing_value(column_name):
    return F.col(column_name).isNull() | (F.trim(F.col(column_name).cast("string")) == "")

def add_error(errors, condition, error_code):
    return F.when(
        condition, F.array_union(errors, F.array(F.lit(error_code)))
    ).otherwise(errors)

Taxi_trip validation rules: 
1. Is dropoff_datetime earlier than pickup_datetime
2. Is trip_distance, passenger_count, fare_amount, total_amount less than 0

Weather validation rules:
1. Is relative humidity between 0–100
2. Is precipitation less than 0
3. Is wind direction between 0–360
4. Is wind speed less than 0
5. Is pressure between 850–1100

Air_quality validation rules:
1. Is latitude between -90 and 90
2. Is longitude between -180 and 180
3. Is sample_measurement negative
4. Is sample_measurement is NaN

Taxi_zones validation rules:
1. Is location_id less than 0

In [15]:
def dataset_rules(dataset_name: str, df: DataFrame):
    if dataset_name == "taxi_trips":
        return [
            ("dropoff_before_pickup", F.col("dropoff_datetime") < F.col("pickup_datetime")),
            ("negative_trip_distance", F.col("trip_distance") < 0),
            ("negative_passenger_count", F.col("passenger_count") < 0),
            ("negative_fare_amount", F.col("fare_amount") < 0),
            ("negative_total_amount", F.col("total_amount") < 0),
            ("invalid_trip_distance", F.isnan("trip_distance")),
            ("invalid_total_amount", F.isnan("total_amount")),
        ]
    if dataset_name == "weather":
        return [
            ("invalid_relative_humidity", (F.col("rhum") < 0) | (F.col("rhum") > 100)),
            ("negative_precipitation", F.col("prcp") < 0),
            ("invalid_wind_direction", (F.col("wdir") < 0) | (F.col("wdir") > 360)),
            ("negative_wind_speed", F.col("wspd") < 0),
            ("invalid_pressure", (F.col("pres") < 850) | (F.col("pres") > 1100)),
        ]
    if dataset_name == "air_quality":
        return [
            ("invalid_latitude", (F.col("latitude") < -90) | (F.col("latitude") > 90)),
            ("invalid_longitude", (F.col("longitude") < -180) | (F.col("longitude") > 180)),
            ("negative_sample_measurement", F.col("sample_measurement") < 0),
            ("invalid_sample_measurement", F.isnan("sample_measurement")),
        ]
    if dataset_name == "taxi_zones":
        return [("invalid_location_id", F.col("location_id") <= 0)]
    raise KeyError(f"No validation rules configured for {dataset_name}")

In [16]:
# Validate the DataFrame against the rules specified in the config and dataset-specific rules
def validate_records(df: DataFrame, dataset_name: str, config: dict) -> DataFrame:
    errors = F.expr("array()").cast("array<string>")
    for column in config["required_columns"]:
        errors = add_error(errors, missing_value(column), f"missing_required:{column}")
    for error_code, condition in dataset_rules(dataset_name, df):
        errors = add_error(errors, condition, error_code)

    primary_key = config["primary_key"]
    duplicate_window = Window.partitionBy(*primary_key).orderBy(
        F.col("_source_file"), F.col("_ingested_at")
    )
    key_is_present = require_all(primary_key)
    validated = df.withColumn("_duplicate_rank", F.row_number().over(duplicate_window))
    errors = add_error(errors, key_is_present & (F.col("_duplicate_rank") > 1), "duplicate_primary_key")
    return validated.withColumn("_validation_errors", errors)

## 6. Delta writers and ingestion metadata

In [17]:
# Delta table schema for ingestion metadata
METADATA_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("dataset_name", StringType(), False),
    StructField("source_format", StringType(), False),
    StructField("source_path", StringType(), False),
    StructField("processed_path", StringType(), False),
    StructField("failed_path", StringType(), False),
    StructField("started_at", TimestampType(), False),
    StructField("completed_at", TimestampType(), False),
    StructField("execution_time_seconds", DoubleType(), False),
    StructField("input_records", LongType(), False),
    StructField("processed_records", LongType(), False),
    StructField("rejected_records", LongType(), False),
    StructField("duplicate_records", LongType(), False),
    StructField("schema_version", StringType(), False),
    StructField("source_schema_json", StringType(), True),
    StructField("output_schema_json", StringType(), True),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True),
])

In [18]:
# Write a DataFrame to a Delta table at the specified path
def write_delta(df: DataFrame, path: str, partition_by=None, mode="overwrite"):
    writer = df.write.format("delta").mode(mode).option("overwriteSchema", "true")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.save(path)

def append_metadata(record: dict):
    metadata_df = spark.createDataFrame([record], schema=METADATA_SCHEMA)
    metadata_df.write.format("delta").mode("append").save(METADATA_PATH)

## 7. Generic ingestion orchestration

The same function handles every dataset. Only configuration, transformation logic, and validation rules vary by dataset.

In [19]:
def ingest_dataset(dataset_name: str, config: dict) -> dict:
    run_id = str(uuid.uuid4())
    started_clock = time.perf_counter()
    started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    input_records = 0
    processed_records = 0
    rejected_records = 0
    duplicate_records = 0
    source_schema_json = None
    output_schema_json = None
    validated = None

    try:
        raw_df = read_source(config) # Read the source data into a DataFrame
        source_schema_json = validate_input_schema(raw_df, config["expected_raw_types"]) # Validate the schema
        input_records = raw_df.count()
        enriched_raw = ( 
            raw_df
            .withColumn("_source_file", F.input_file_name())
            .withColumn("_ingestion_run_id", F.lit(run_id))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_schema_version", F.lit(config["schema_version"]))
        )
        transformed = TRANSFORMERS[config["transformer"]](enriched_raw) # Transform the DataFrame using the dataset-specific transformer
        output_schema_json = transformed.schema.json()
        validated = validate_records(transformed, dataset_name, config).persist(StorageLevel.MEMORY_AND_DISK) # Validate the records and persist the DataFrame in memory and disk
        
        # Filtering out invalid records
        processed_df = ( 
            validated.filter(F.size("_validation_errors") == 0)
            .drop("_validation_errors", "_duplicate_rank")
        )
        failed_df = validated.filter(F.size("_validation_errors") > 0).drop("_duplicate_rank") 

        processed_records = processed_df.count()
        rejected_records = failed_df.count()
        duplicate_records = failed_df.filter(
            F.array_contains("_validation_errors", "duplicate_primary_key")
        ).count()
        if input_records != processed_records + rejected_records:
            raise AssertionError("Input count does not equal processed plus rejected records")

        # Write the processed and failed DataFrames to Delta tables
        write_delta(processed_df, config["processed_path"], config["partition_by"])
        write_delta(failed_df, config["failed_path"])
        status = "SUCCESS"
        error_message = None
    except Exception as exc:
        status = "FAILED"
        error_message = f"{type(exc).__name__}: {exc}"
    finally:
        # Record the ingestion run metadata
        completed_at = datetime.now(timezone.utc).replace(tzinfo=None)
        elapsed = time.perf_counter() - started_clock
        record = {
            "run_id": run_id,
            "dataset_name": dataset_name,
            "source_format": config["format"],
            "source_path": config["source_path"],
            "processed_path": config["processed_path"],
            "failed_path": config["failed_path"],
            "started_at": started_at,
            "completed_at": completed_at,
            "execution_time_seconds": float(elapsed),
            "input_records": int(input_records),
            "processed_records": int(processed_records),
            "rejected_records": int(rejected_records),
            "duplicate_records": int(duplicate_records),
            "schema_version": config["schema_version"],
            "source_schema_json": source_schema_json,
            "output_schema_json": output_schema_json,
            "status": status,
            "error_message": error_message,
        }
        append_metadata(record)
        if validated is not None:
            validated.unpersist()

    if status == "FAILED":
        raise RuntimeError(f"Ingestion failed for {dataset_name}: {error_message}")
    return record

## 8. Run all four ingestions

In [20]:
RUN_ORDER = ["taxi_zones", "weather", "taxi_trips", "air_quality"]
ingestion_results = []
for dataset_name in RUN_ORDER:
    print(f"\nIngesting {dataset_name} ...")
    result = ingest_dataset(dataset_name, DATASETS[dataset_name])
    ingestion_results.append(result)
    print(json.dumps({
        key: result[key]
        for key in [
            "dataset_name", "input_records", "processed_records",
            "rejected_records", "duplicate_records",
            "execution_time_seconds", "schema_version", "status",
        ]
    }, indent=2))


Ingesting taxi_zones ...


26/09/10 11:36:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


{
  "dataset_name": "taxi_zones",
  "input_records": 265,
  "processed_records": 265,
  "rejected_records": 0,
  "duplicate_records": 0,
  "execution_time_seconds": 6.583667875034735,
  "schema_version": "1.0.0",
  "status": "SUCCESS"
}

Ingesting weather ...
{
  "dataset_name": "weather",
  "input_records": 8784,
  "processed_records": 8783,
  "rejected_records": 1,
  "duplicate_records": 1,
  "execution_time_seconds": 3.2497132499702275,
  "schema_version": "1.0.0",
  "status": "SUCCESS"
}

Ingesting taxi_trips ...


26/09/10 11:36:14 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/raw/yellow_tripdata_2024-*.parquet.
java.io.FileNotFoundException: File /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/raw/yellow_tripdata_2024-*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSourc

{
  "dataset_name": "taxi_trips",
  "input_records": 9554778,
  "processed_records": 9417689,
  "rejected_records": 137089,
  "duplicate_records": 1,
  "execution_time_seconds": 199.22633941611275,
  "schema_version": "1.0.0",
  "status": "SUCCESS"
}

Ingesting air_quality ...


26/09/10 11:39:42 ERROR CodeGenerator: Failed to compile the generated Java code.
org.codehaus.commons.compiler.InternalCompilerException: Compiling "GeneratedClass" in File 'generated.java', Line 1, Column 1: File 'generated.java', Line 26, Column 16: Compiling "processNext()"
	at org.codehaus.janino.UnitCompiler.compile2(UnitCompiler.java:402)
	at org.codehaus.janino.UnitCompiler.access$000(UnitCompiler.java:236)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:363)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:361)
	at org.codehaus.janino.Java$CompilationUnit.accept(Java.java:371)
	at org.codehaus.janino.UnitCompiler.compileUnit(UnitCompiler.java:361)
	at org.codehaus.janino.SimpleCompiler.cook(SimpleCompiler.java:264)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:294)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:288)
	at org.codehaus.janino.ClassBodyEvaluator.cook(Class

{
  "dataset_name": "air_quality",
  "input_records": 8139551,
  "processed_records": 7927367,
  "rejected_records": 212184,
  "duplicate_records": 907,
  "execution_time_seconds": 32.59890004200861,
  "schema_version": "1.0.0",
  "status": "SUCCESS"
}


## 9. Verify processed, failed, partition, and metadata outputs

In [21]:
# Print summary of ingestion results
for dataset_name in RUN_ORDER:
    config = DATASETS[dataset_name]
    processed_df = spark.read.format("delta").load(config["processed_path"])
    failed_df = spark.read.format("delta").load(config["failed_path"])
    print("=" * 80)
    print(dataset_name)
    print("processed rows:", processed_df.count())
    print("failed rows:", failed_df.count())
    processed_df.printSchema()
    processed_df.show(3, truncate=False)
    failed_df.select("_validation_errors").groupBy("_validation_errors").count().orderBy(F.desc("count")).show(20, truncate=False)

print("Taxi partitions:")
spark.read.format("delta").load(DATASETS["taxi_trips"]["processed_path"]).select(
    "pickup_year", "pickup_month"
).distinct().orderBy("pickup_year", "pickup_month").show(100, truncate=False)

print("Air Quality partitions:")
spark.read.format("delta").load(DATASETS["air_quality"]["processed_path"]).select(
    "observation_year", "observation_month"
).distinct().orderBy("observation_year", "observation_month").show(100, truncate=False)

taxi_zones
processed rows: 265
failed rows: 0
root
 |-- location_id: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- service_zone: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_run_id: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _schema_version: string (nullable = true)

+-----------+---------+-----------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------+--------------------------+---------------+
|location_id|borough  |zone                   |service_zone|_source_file                                                                                                                                    |_ingestion_run_id                   |_ingested_at              |_schema_version|
+-----------+---------+---

In [ ]:
# Print ingestion metadata
ingestion_metadata = spark.read.format("delta").load(METADATA_PATH)
ingestion_metadata.select(
    "run_id", "dataset_name", "started_at", "completed_at",
    "execution_time_seconds", "input_records", "processed_records",
    "rejected_records", "duplicate_records", "schema_version",
    "status", "error_message",
).orderBy(F.desc("started_at")).show(50, truncate=False)

+------------------------------------+------------+--------------------------+--------------------------+----------------------+-------------+-----------------+----------------+-----------------+--------------+-------+-------------+
|run_id                              |dataset_name|started_at                |completed_at              |execution_time_seconds|input_records|processed_records|rejected_records|duplicate_records|schema_version|status |error_message|
+------------------------------------+------------+--------------------------+--------------------------+----------------------+-------------+-----------------+----------------+-----------------+--------------+-------+-------------+
|7285a252-a1cd-44f3-aa89-6b6c7dd9f71d|air_quality |2026-09-10 03:39:34.201217|2026-09-10 03:40:06.799456|32.59890004200861     |8139551      |7927367          |212184          |907              |1.0.0         |SUCCESS|NULL         |
|48a508d2-35dc-41fd-af72-6b384170120b|taxi_trips  |2026-09-10 03:36:

26/09/10 16:18:41 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 450180 ms exceeds timeout 120000 ms
26/09/10 16:18:41 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/10 16:18:43 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora